In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# ------------------------- config
N = 10_000                    # simulated trajectories
N_STEPS, DT, K = 800, 0.1, 20
START, TARGET = np.array([0.0, 0.0]), np.array([10.0, 10.0])
DRIFT, VOL = 0.5, 1

LAMBDA = np.array([1.0, 1.0, 1.0])       # quality coefficients (D, P, H)
BETA_BOUNDS = (0.1, 200.0)
BLOCK = 512
X0 = np.array([0.33, 0.33, 0.33, 0.0])      # start: w = (0.33, 0.33, 0.33)
SEED = 42
rng = np.random.default_rng()


# ------------------------------ simulation + features
def wrap_angle(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


def simulate_features():
    pos = np.tile(START, (N, 1))
    d0 = np.maximum(np.linalg.norm(TARGET - pos, axis=1), 1e-8)
    sum_dist, path_len, h_sum, n_blocks = np.ones(N), np.zeros(N), np.zeros(N), 0
    ckpt = pos.copy()
    for t in range(1, N_STEPS + 1):
        vec = TARGET - pos
        dist = np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-12)
        new = pos + DRIFT * vec / dist * DT + VOL * np.sqrt(DT) * rng.standard_normal((N, 2))
        path_len += np.linalg.norm(new - pos, axis=1)
        sum_dist += np.linalg.norm(TARGET - new, axis=1) / d0
        pos = new
        if t % K == 0:
            disp, goal = pos - ckpt, TARGET - ckpt
            err = wrap_angle(np.arctan2(goal[:, 1], goal[:, 0]) - np.arctan2(disp[:, 1], disp[:, 0]))
            h_sum += np.abs(err) / np.pi
            n_blocks += 1
            ckpt = pos.copy()
    d_final = np.linalg.norm(TARGET - pos, axis=1)
    return (sum_dist / (N_STEPS + 1),                      # D  lower = better
            (d0 - d_final) / np.maximum(path_len, 1e-8),   # P  higher = better
            h_sum / n_blocks)                              # H  lower = better


print("Simulating trajectories...")



D, P, H = simulate_features()


# ======================================================================
# FEATURE ENGINEERING
# ======================================================================

N_FEATURES = 6
SIGN = np.array([-1, 1, -1, -1, -1, -1])

F = np.column_stack([
    D,
    P,
    H,
    D**2,
    H**2,
    D * H
])

assert F.shape == (N, N_FEATURES)

print("Feature matrix shape:", F.shape)


# ======================================================================
# TRAIN / TEST SPLIT
# ======================================================================

perm = rng.permutation(N)

split = int(0.8 * N)

tr = perm[:split]
te = perm[split:]

F_tr = F[tr]
F_te = F[te]

# Fit normalization on training data.
mu = F_tr.mean(axis=0)

sd = np.maximum(
    F_tr.std(axis=0),
    1e-12
)

Z_tr = (F_tr - mu) / sd
Z_te = (F_te - mu) / sd

# Sign-adjust standardized features.
S_tr = Z_tr * SIGN
S_te = Z_te * SIGN


# ======================================================================
# QUALITY FUNCTION
# ======================================================================

def quality(Z):

    return (
        -Z[:, 0]
        + Z[:, 1]
        - Z[:, 2]
        - Z[:, 3]
        - Z[:, 4]
        - Z[:, 5]
    )


def sort_best_first(S, Z):
    """Sort trajectories from highest to lowest quality."""

    order = np.argsort(-quality(Z))

    return S[order]


S_tr = sort_best_first(S_tr, Z_tr)[:, :3]
S_te = sort_best_first(S_te, Z_te)[:, :3]
n_tr = len(S_tr)
print(f"train items {n_tr:,}  ->  {n_tr * (n_tr - 1) // 2:,} unique training pairs")


# ------------------------------- exact all-pairs loss + gradient
def allpairs_loss_grad(x, S, block=BLOCK):
    """x = [w_D, w_P, w_H, log beta].  Returns L and dL/dx over all i<j pairs of sorted S."""
    w, beta = x[:3], np.exp(x[3])
    n = len(S)
    score = S @ w
    total = n * (n - 1) / 2.0
    loss = 0.0
    g_w = np.zeros(3)
    g_lb = 0.0

    for i0 in range(0, n, block):
        i1 = min(i0 + block, n)
        si, sj = score[i0:i1], score[i0:]
        m = si[:, None] - sj[None, :]
        mask = np.arange(sj.size)[None, :] > np.arange(i1 - i0)[:, None]   # keep j > i only
        z = -beta * m
        loss += (np.logaddexp(0.0, z) * mask).sum()
        sig = (0.5 * (1.0 + np.tanh(0.5 * z))) * mask
        g_lb += (sig * z).sum()
        g_w += -beta * (sig.sum(axis=1) @ S[i0:i1] - sig.sum(axis=0) @ S[i0:])

    return loss / total, np.append(g_w, g_lb) / total


# ----------------------------------- optimization (SLSQP)
path = [X0.copy()]


def callback(x):
    path.append(x.copy())
    print(f"  iter {len(path) - 1:3d} | w = ({x[0]:.5f}, {x[1]:.5f}, {x[2]:.5f}) "
          f"| beta = {np.exp(x[3]):8.3f}")


print("\nSLSQP on the exact all-pairs Bradley-Terry loss...")
res = minimize(
    allpairs_loss_grad, X0, args=(S_tr,), jac=True, method="SLSQP",
    bounds=[(0, 1)] * 3 + [tuple(np.log(BETA_BOUNDS))],
    constraints={"type": "eq", "fun": lambda x: x[:3].sum() - 1.0,
                 "jac": lambda x: np.array([1.0, 1.0, 1.0, 0.0])},
    callback=callback,
    options={"maxiter": 200, "ftol": 1e-12},
)
theta = res.x[:3]
beta = float(np.exp(res.x[3]))


# ---------------------------------------- results
print("\n" + "=" * 62)
print("OPTIMIZED REWARD WEIGHTS")
print("=" * 62)
print("success:", res.success, "|", res.message)
print(f"w_D = {theta[0]:.6f}")
print(f"w_P = {theta[1]:.6f}")
print(f"w_H = {theta[2]:.6f}")
print(f"sum = {theta.sum():.6f}")
print(f"beta* = {beta:.3f}" + ("   (at upper bound: preferences look noiseless, "
                              "only the direction of w is meaningful)" if beta > 0.98 * BETA_BOUNDS[1] else ""))
print(f"train all-pairs loss = {res.fun:.8f}")

alpha = beta * theta / sd[:3]
print("\nReward on z-scored features:  G = beta*( -w_D z(D) + w_P z(P) - w_H z(H) )")
print(f"  = -{beta * theta[0]:.4f} z(D) + {beta * theta[1]:.4f} z(P) - {beta * theta[2]:.4f} z(H)")
print("Equivalent raw-feature coefficients:  G = -aD*D + aP*P - aH*H + const")
print(f"  aD = {alpha[0]:.5f}   aP = {alpha[1]:.5f}   aH = {alpha[2]:.5f}")

# held-out fit check (all test pairs)
Lte, _ = allpairs_loss_grad(res.x, S_te)
sc = S_te @ theta
n_te = len(S_te)
correct = sum(np.count_nonzero(sc[i, None] > sc[i + 1:]) for i in range(n_te - 1))
print(f"\nheld-out: all-pairs loss = {Lte:.6f} | pairwise accuracy = "
      f"{correct / (n_te * (n_te - 1) / 2):.4f}")

Simulating trajectories...
Feature matrix shape: (10000, 6)
train items 8,000  ->  31,996,000 unique training pairs

SLSQP on the exact all-pairs Bradley-Terry loss...
  iter   1 | w = (0.39058, 0.20176, 0.40767) | beta =    1.194
  iter   2 | w = (0.45704, 0.00000, 0.54296) | beta =    2.359
  iter   3 | w = (0.41940, 0.00000, 0.58060) | beta =    6.324
  iter   4 | w = (0.29018, 0.34796, 0.36186) | beta =   23.341
  iter   5 | w = (0.35760, 0.64240, 0.00000) | beta =  146.076
  iter   6 | w = (0.49385, 0.29774, 0.20841) | beta =   30.513
  iter   7 | w = (0.44928, 0.21073, 0.33999) | beta =   23.308
  iter   8 | w = (0.44012, 0.17507, 0.38481) | beta =   19.160
  iter   9 | w = (0.42654, 0.17736, 0.39611) | beta =   24.300
  iter  10 | w = (0.44369, 0.16674, 0.38957) | beta =   52.121
  iter  11 | w = (0.43839, 0.17563, 0.38598) | beta =   86.960
  iter  12 | w = (0.43919, 0.16932, 0.39149) | beta =  116.026
  iter  13 | w = (0.44368, 0.17164, 0.38468) | beta =  117.167
  iter  14 | 